<img src="http://imgur.com/1ZcRyrc.png" style="float: left; margin: 20px; height: 120px">

# Evocative of One Another: Dungeons & Dragons and Pathfinder RPG
### (Fill out this cell as the project progresses, then move to README.md as the technical report.)

*Deval Mehta*

## Table of Contents
1) [Overview](#Overview) 
2) [Data Dictionary](<#Data Dictionary>)
3) [Requirements](#Requirements)
4) [Executive Summary](#Executive-Summary)
    1) [The Data](<#The Deta>)
    2) [Baseline Values](<#Baseline Values>)
    3) [Data Transformation](<#Data Transformation>)
    4) [Model Selection Criteria](<#Model Selection Criteria>)
    5) [Analysis](#Analysis)
    6) [Implications](#Implications)
    7) [Next Steps](#Next-Steps)

## Overview
Tabletop Role Playing Games (TTRPGs) experienced a massive popularity boom in the latter half of the 2010s due to the advent of live-streaming in-person games and online gaming communities, which has continued into the present day. Two giants stand atop the high-fantasy TTRPG genre: *Dungeons & Dragons* and *Pathfinder*. The two systems are strikingly similar -- so much so that upon its inception in 2009, Pathfinder was dubbed by fans as "D&D 3.75." Among other things, the two systems share relatively similar settings, rules, and character classes. With so much in common, one might wonder from the perspective of a player or game master: are the two truly distinct enough to be considered separate systems? Our goal is to consider this question through the lens of natural language processing by analyzing posts on the subreddits r/DungeonsAndDragons and r/Pathfinder_RPG. We've retrieved nearly 2,000 posts from each subreddit using the Python Reddit API Wrapper (PRAW) to train and test our models. We generate an incredible number of features from our data by considering 1,2, and 3-grams, explore some factors that might distinguish posts on one subreddit from those of another, and finally compare the logistic regression, random forest, and support vector machine approaches to the problem.

**(Insert brief results summary here)**

## Data Dictionary
Rather than enumerating the numerous tokens of each document in our corpus, we summarize the information collected from each Reddit post.

| Information | Data Type | Description | Notes |
|---|---|---|---|
| id | `string` | The individual ID number assigned to each post by Reddit | We use this to remove any possible duplicates. |
| created_utc | `float64` | The number of seconds that elapsed between 1970-Jan-01 00:00 UTC and the creation of the post. | |
| title | `string` | The title of each Reddit post | |
| self_text | `string` | The body text of each Reddit post | |
| subreddit | `string` | The subreddit from which each post was retrieved | |

To conduct our analysis, we concatenate the `title` and `self_text` to create a new `all_text` column, which we tokenized. In addition, we created a binarized version of `subreddit` called `isDnD`, which is an integer representation of whether a post belongs to r/DungeonsAndDragons. This is done to ensure our `LogisticRegression` object will be able to interpret the two categories.

## Requirements

| Library | Module | Purpose |
|---|---|---|

## Executive Summary
### The Data

### Baseline Values

### Data Transformation

### Model Selection Criteria

### Analysis

### Implications

### Next Steps


### Imports

In [1]:
# Staple imports for data handling, analysis, and calculation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# NLP imports
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet, stopwords
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Imports for model instantiation, fitting, and scoring
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Data tranformation and preprocessing imports
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

# Reddit Scraper
import praw
from reddit_scraper import *

### Load the Data in
We only needed to run this cell one time. The "new" data is current as of 2024-11-21.

In [10]:
# Retrieve top 1000 posts of all time from r/DungeonsAndDragons and r/Pathfinder_RPG
# unified_data('DungeonsAndDragons', 'all', 1000, 1000)
# unified_data('Pathfinder_RPG', 'all', 1000, 1000)

### Inspect and Preprocess the Data

Read in the data from the `.csv` files generated by our script `reddit_scraper.py`. The script takes the top 1000 posts of all time and the newest 1000 posts from each of our given subreddits, concatenates the listings, and removes any duplicate entries before writing out to a `.csv` file.

In [2]:
dnd = pd.read_csv('../data/DungeonsAndDragons.csv')
pathfinder = pd.read_csv('../data/Pathfinder_RPG.csv')

display(dnd.head())
display(pathfinder.head())

,Unnamed: 0,id,created_utc,title,self_text,subreddit
0,0,7yx5bd,1.519144e+09,When you confuse Wisdom with Intelligence,NaN,DungeonsAndDragons
1,1,kutjp1,1.610334e+09,"Live out your fantasy this way, and you will b...",NaN,DungeonsAndDragons
2,2,7t9g94,1.517018e+09,Escape From Flavortown,NaN,DungeonsAndDragons
3,3,1guf7p9,1.731963e+09,"My patient said to me, “You must be into sport...",NaN,DungeonsAndDragons
4,4,7nvhag,1.514995e+09,Persuasive Bard gets persuasive...,NaN,DungeonsAndDragons


,Unnamed: 0,id,created_utc,title,self_text,subreddit
0,0,7enkie,1.511319e+09,Love having everything you need for Pathfinder...,NaN,Pathfinder_RPG
1,1,10adsp3,1.673565e+09,Paizo Announces System-Neutral Open RPG License,NaN,Pathfinder_RPG
2,2,11b1br0,1.677271e+09,Thanks for playing Pathfinder.,We appreciate you.,Pathfinder_RPG
3,3,10dtypy,1.673907e+09,Newcomers; Archives of Nethys is free. It's no...,"Seriously, I've seen so many people lately com...",Pathfinder_RPG
4,4,10mlx2w,1.674829e+09,Pathfinder sales are through the roof followin...,NaN,Pathfinder_RPG


Assess the number of posts we have accessed. PRAW does not meet the full requested amount in a given pull. Some entries may also have been deleted, so we will not have a full 2000 posts from each subreddit.

In [3]:
dnd.shape, pathfinder.shape

((1973, 6), (1993, 6))

With 1,973 D&D posts and 1,993 Pathfinder posts, we see that our classes will be fairly balanced (nearly perfectly so). For data exploration and modeling, this means we will not have to worry about the `stratify` argument when we train-test split and we will not have to weight classes when attempting a logistic regression.

In [4]:
dnd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1973 entries, 0 to 1972
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   1973 non-null   int64  
 1   id           1973 non-null   object 
 2   created_utc  1973 non-null   float64
 3   title        1973 non-null   object 
 4   self_text    653 non-null    object 
 5   subreddit    1973 non-null   object 
dtypes: float64(1), int64(1), object(4)
memory usage: 92.6+ KB


In [5]:
pathfinder.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1993 entries, 0 to 1992
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   1993 non-null   int64  
 1   id           1993 non-null   object 
 2   created_utc  1993 non-null   float64
 3   title        1993 non-null   object 
 4   self_text    1635 non-null   object 
 5   subreddit    1993 non-null   object 
dtypes: float64(1), int64(1), object(4)
memory usage: 93.5+ KB


Most of the D&D data has no body text and we lack a decent amount from the Pathfinder data as well. As such, we plan to consider the titles and body text for our analysis. Let us create a new feature which combines them.

In [7]:
# Fill N/As with empty strings so that we can concatenate
dnd.fillna('', inplace = True)
pathfinder.fillna('', inplace = True)

# Create new columns to pull features from which will serve as our corpora
dnd['all_text'] = dnd['title'] + ' ' +  dnd['self_text']
pathfinder['all_text'] = pathfinder['title'] + ' ' + pathfinder['self_text']

display(dnd.head()), display(pathfinder.head());

,Unnamed: 0,id,created_utc,title,self_text,subreddit,all_text
0,0,7yx5bd,1.519144e+09,When you confuse Wisdom with Intelligence,,DungeonsAndDragons,When you confuse Wisdom with Intelligence
1,1,kutjp1,1.610334e+09,"Live out your fantasy this way, and you will b...",,DungeonsAndDragons,"Live out your fantasy this way, and you will b..."
2,2,7t9g94,1.517018e+09,Escape From Flavortown,,DungeonsAndDragons,Escape From Flavortown
3,3,1guf7p9,1.731963e+09,"My patient said to me, “You must be into sport...",,DungeonsAndDragons,"My patient said to me, “You must be into sport..."
4,4,7nvhag,1.514995e+09,Persuasive Bard gets persuasive...,,DungeonsAndDragons,Persuasive Bard gets persuasive...


,Unnamed: 0,id,created_utc,title,self_text,subreddit,all_text
0,0,7enkie,1.511319e+09,Love having everything you need for Pathfinder...,,Pathfinder_RPG,Love having everything you need for Pathfinder...
1,1,10adsp3,1.673565e+09,Paizo Announces System-Neutral Open RPG License,,Pathfinder_RPG,Paizo Announces System-Neutral Open RPG License
2,2,11b1br0,1.677271e+09,Thanks for playing Pathfinder.,We appreciate you.,Pathfinder_RPG,Thanks for playing Pathfinder. We appreciate you.
3,3,10dtypy,1.673907e+09,Newcomers; Archives of Nethys is free. It's no...,"Seriously, I've seen so many people lately com...",Pathfinder_RPG,Newcomers; Archives of Nethys is free. It's no...
4,4,10mlx2w,1.674829e+09,Pathfinder sales are through the roof followin...,,Pathfinder_RPG,Pathfinder sales are through the roof followin...


Before proceeding, we should ensure that there are no null values among `all_text`

In [8]:
dnd['all_text'].isnull().sum(), pathfinder['all_text'].isnull().sum()

(0, 0)

Now we can combine our two corpora into a single corpus, which we will analyze and feed to our models. If we are successful, our new corpus will contain 3,966 documents.

In [19]:
combined_df = pd.concat([dnd.reset_index(drop = True), pathfinder.reset_index(drop = True)], ignore_index = True)

# ignore_index() is persistently not working, so add the following line to remove the annoying index preserving column
combined_df = combined_df.drop(columns = 'Unnamed: 0')
combined_df.shape

(3966, 6)

In [20]:
combined_df.columns

Index(['id', 'created_utc', 'title', 'self_text', 'subreddit', 'all_text'], dtype='object')

We need one more new column: a binarized version of `subreddit`. We will create a new column called `isDnD` which will be 1 if the post comes from r/DungeonsAndDragons and 0 if the post comes from r/Pathfinder_RPG. We maintain the column `subreddit`, rather than simply mapping it, so that we may use it for our Random Forest and Support Vector Machine models.

In [24]:
combined_df['isDnD'] = (combined_df['subreddit'] == 'DungeonsAndDragons').astype(int)
combined_df.head()

,id,created_utc,title,self_text,subreddit,all_text,isDnD
0,7yx5bd,1.519144e+09,When you confuse Wisdom with Intelligence,,DungeonsAndDragons,When you confuse Wisdom with Intelligence,1
1,kutjp1,1.610334e+09,"Live out your fantasy this way, and you will b...",,DungeonsAndDragons,"Live out your fantasy this way, and you will b...",1
2,7t9g94,1.517018e+09,Escape From Flavortown,,DungeonsAndDragons,Escape From Flavortown,1
3,1guf7p9,1.731963e+09,"My patient said to me, “You must be into sport...",,DungeonsAndDragons,"My patient said to me, “You must be into sport...",1
4,7nvhag,1.514995e+09,Persuasive Bard gets persuasive...,,DungeonsAndDragons,Persuasive Bard gets persuasive...,1


With the data combined, we can isolate the text and normalize it. In particular, we will convert all the text in our documents to We will preserve the id, token, subreddit, and isDnD columns for further analysis. Since sentiment analysis reads full documents, we preserve the original dataframe separately for that purpose.

### Data Exploration

#### Title and Post Length in Words

#### Sentiment Analysis

#### Most and Least Frequent n-grams

In [ ]:
#### 

### Establish the baseline

In [13]:
dnd['self_text'].isnull().sum(), pathfinder['self_text'].isnull().sum()

(1320, 358)

### Set up a pipeline to transform the data

In [ ]:
# Vecgorize the text data (will use title, since so many posts don't have bodies)
# Map subreddits to 0 and 1
# Fit and transform *some* models to the vectorized data
# Do some kind of hyperparameter tuning (GridSearchCV or RandomizedSearchCV)

### Model Selection & Explanation

### Fit and Test the Model

### Model Analysis

In [ ]:
# Sentiment Analysis perhaps
# Model Evaluation
# Hyperparamater Search Visualization
# ROC Curve/AUC

### Tweak and Repeat